# DataHub Error Case 04 — Kafka Consumer Error

**Monitor:** DataHub Kafka Consumer Error  
**Component:** datahub-mae-consumer  
**Note:** Self-healing error — consumer always rejoins. Threshold is count > 5.

In [ ]:
import sys
sys.path.insert(0, '.')
import importlib
import datahub_test_utils
importlib.reload(datahub_test_utils)
from datahub_test_utils import *

kubectl('get pods -n datahub-hynix | grep consumer')

In [ ]:
# Cell 2: Restart MAE consumer to trigger Kafka rebalance
print('Restarting MAE consumer...')
kubectl('rollout restart deployment/datahub-datahub-mae-consumer -n datahub-hynix')
time.sleep(5)
kubectl('get pods -n datahub-hynix | grep mae-consumer')

In [ ]:
# Cell 3: Find consumer error logs
wait_for_logs(20)
print('Searching for Kafka consumer errors (last 2 min)...')
hits = search_datahub_kafka_errors(minutes_ago=2)
print_logs(hits)

In [ ]:
# Cell 4: Verify self-recovery
print('Waiting for consumer to rejoin group...')
time.sleep(30)
kubectl('get pods -n datahub-hynix | grep mae-consumer')
r = health_check()
print(f'GMS health: {r.status_code}')

print('\n=== Summary ===')
print('Error:   Offset commit failure after rebalance')
print('Monitor: Kafka Consumer Error fires when count > 5')
print('Fix:     Self-healing — consumer rejoins automatically')